# Issue #31: check `predict.py` on Colab

**Runtime > Run all.** The only manual step is the Google Drive sign-in pop-up.

This runs every `predict_check.py` check in one go. Acceptance rule for each run: exact chunk match, mean |diff| <= 1e-3,
max |diff| <= 5e-2 against the saved validation probabilities, and zero yes/no flips at the cutoffs.
0. **Cold model load** of both runs (run in a fresh runtime so the first load is really cold).
1. **GPU, bf16** (B1): batch sizes 8 and 1, plus how much the output moves between them.
1b. **Exact training batching** (diagnostic): all 1,841 val chunks in training's batches; also reports the strict max |diff| <= 1e-3.
1c. **GPU, fp32**: batch sizes 8 and 1, to see how much fp32 output moves with batching.
1d. **GPU fp32 vs CPU fp32**, compared directly.
2. **CPU, fp32** (B2) on the same machine (`--device cpu`, so no runtime switch), with CPU timings.

Every run does check A (chunk boundaries, raw offsets); runs 1, 1c and 2 also do the smoke test C on an unseen contract.


Needs: an **A100 or L4** GPU (bf16; a T4 falls back to fp32 and can't run B1), and the `cuad_finetune` folder in your Drive
(if it was shared with you, add a shortcut to it in My Drive). Optional: upload a non-CUAD contract to `/content/my_contract.txt`
for the smoke test; otherwise it uses the text of `test.json` contract 0 (text only, no labels, no metrics).

In [1]:
# settings
BRANCH = "albert-issue31"
REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1N-contract-review-challenge.git"
DRIVE_DIR = "/content/drive/MyDrive/cuad_finetune"
MODEL_ROOT = f"{DRIVE_DIR}/runs"                         # holds <run>/model/ for both ensemble runs
CHUNKS_DF = f"{DRIVE_DIR}/data/chunks_df.parquet"        # training chunks, for the exact chunk text/offset check
SMOKE_TEXT = "/content/my_contract.txt"                  # optional unseen contract

In [2]:
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 or L4"
name, (major, minor) = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute capability {major}.{minor})")
if major < 8:
    print("WARNING: this GPU has no fast bf16, so the GPU run will use fp32 and the 1e-3 acceptance test (B1) won't "
          "apply. Switch to an A100 or L4 for the real check.")

GPU: NVIDIA L4 (compute capability 8.9)


In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
for path in [Path(MODEL_ROOT) / "modernbert_v2_biasinit" / "model" / "config.json",
             Path(MODEL_ROOT) / "modernbert_v1_6ep" / "model" / "config.json",
             Path(CHUNKS_DF)]:
    print(("found    " if path.exists() else "MISSING  ") + str(path))

Mounted at /content/drive
found    /content/drive/MyDrive/cuad_finetune/runs/modernbert_v2_biasinit/model/config.json
found    /content/drive/MyDrive/cuad_finetune/runs/modernbert_v1_6ep/model/config.json
found    /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet


In [4]:
import os
REPO_DIR = "/content/repo"
if not os.path.exists(REPO_DIR):
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --quiet origin {BRANCH} && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} reset --quiet --hard origin/{BRANCH}
!git -C {REPO_DIR} log --oneline -1
!pip install -q git+https://github.com/brandonstarxel/chunking_evaluation.git backoff
%cd {REPO_DIR}/notebooks

f6d0b15 (HEAD -> albert-issue31, origin/albert-issue31) Enhance predict.py and predict_check.py for precision handling and device comparison checks
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, bu

In [5]:
import subprocess, sys, time

env = {**os.environ, "CUAD_MODEL_ROOT": MODEL_ROOT}
smoke = ["--smoke-text", SMOKE_TEXT] if os.path.exists(SMOKE_TEXT) else ["--smoke-test-json", "0"]
chunks = ["--chunks-df", CHUNKS_DF] if os.path.exists(CHUNKS_DF) else []
LOG_DIR = Path("/content/predict_check_logs")
LOG_DIR.mkdir(exist_ok=True)


def run_check(label, extra_args):
    # streams predict_check.py's output into the cell and keeps a copy in LOG_DIR
    cmd = [sys.executable, "predict_check.py", *chunks, *smoke, *extra_args]
    print("$ " + " ".join(cmd), flush=True)
    start = time.time()
    log = LOG_DIR / f"{label}.txt"
    with open(log, "w") as f:
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end="")
                f.write(line)
    print()
    print(f"[{label}] exit code {proc.returncode} after {(time.time() - start) / 60:.1f} min, log: {log}")
    return proc.returncode


results = {}

### 0. Cold model load (both runs). Only cold in a **fresh runtime**: Runtime > Disconnect and delete runtime, then Run all

In [6]:
results["Model load timing"] = run_check("load_timing", ["--time-load"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --time-load
torch 2.14.1+cu130, transformers 5.22.1; device cuda (NVIDIA L4). Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1

=== model load timing: 2 runs from /content/drive/MyDrive/cuad_finetune/runs (1,204 MB on disk) to cuda ===

Loading weights: 100%|██████████| 138/138 [00:09<00:00, 14.47it/s]

Loading weights: 100%|██████████| 138/138 [00:08<00:00, 15.83it/s]

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 2364.85it/s]

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 2637.27it/s]
  first load (cold only in a fresh runtime)                  46.24s
  reload, files already read once (OS/Drive cache warm)       1.11s
  cached call (models already in memory)                      0.00s
  (excludes importing torch/transformers; tokenizer and chunker load separately on the first predict_contract call)

[load_timing] exit code 0 afte

### 1. GPU, bf16 autocast (B1, acceptance), batch sizes 8 and 1

In [7]:
results["GPU bf16 (B1)"] = run_check("gpu_bf16", ["--batch-sizes", "8", "1"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --batch-sizes 8 1
torch 2.14.1+cu130, transformers 5.22.1; device cuda (NVIDIA L4). Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AG

### 1b. Exact training batching (diagnostic): all 1,841 val chunks, batches of 16 in `val_probs` row order, bf16

In [8]:
results["GPU bf16 exact batching"] = run_check("gpu_bf16_exact", ["--exact-batching"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --exact-batching
torch 2.14.1+cu130, transformers 5.22.1; device cuda (NVIDIA L4). Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGR

### 1c. GPU fp32, batch sizes 8 and 1: how much fp32 output moves with batching

In [9]:
results["GPU fp32"] = run_check("gpu_fp32", ["--precision", "fp32", "--batch-sizes", "8", "1"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --precision fp32 --batch-sizes 8 1
torch 2.14.1+cu130, transformers 5.22.1; device cuda (NVIDIA L4). Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.

### 1d. GPU fp32 vs CPU fp32, compared directly on the same 120 chunks

In [10]:
results["GPU vs CPU fp32"] = run_check("gpu_vs_cpu_fp32", ["--compare-devices"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --compare-devices
torch 2.14.1+cu130, transformers 5.22.1; device cuda (NVIDIA L4). Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AG

### 2. CPU, fp32 (B2), on this machine's CPU. Slower: a few minutes for the 3 contracts with 2 models

In [11]:
results["CPU fp32 (B2)"] = run_check("cpu_fp32", ["--device", "cpu", "--precision", "fp32"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --device cpu --precision fp32
torch 2.14.1+cu130, transformers 5.22.1; device cpu. Runs trained/predicted on A100 with 2.11.0+cu130 / 5.20.1
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGR

### Summary

In [12]:
for label, code in results.items():
    print(f"{label:16s} {'PASS' if code == 0 else 'FAIL (see its output above)'}")
print()
print(f"logs in {LOG_DIR}: " + ", ".join(p.name for p in sorted(LOG_DIR.glob('*.txt'))))
assert all(code == 0 for code in results.values()), "at least one check failed"

Model load timing PASS
GPU bf16 (B1)    PASS
GPU bf16 exact batching PASS
GPU fp32         PASS
GPU vs CPU fp32  PASS
CPU fp32 (B2)    PASS

logs in /content/predict_check_logs: cpu_fp32.txt, gpu_bf16.txt, gpu_bf16_exact.txt, gpu_fp32.txt, gpu_vs_cpu_fp32.txt, load_timing.txt
